In [1]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from flax import nnx

# from hcnn.environment import Environment
from hcnn.optimization import OptimizationSolverShallow
from hcnn.policy.policy_mlp import PolicyMLP
from hcnn.projection import ProjectionNoSubstitutionNoTerminal
from hcnn.system_constraints import SystemConstraintsBox
from hcnn.system_cost import SystemCostQuadratic
from hcnn.system_dynamics import SystemDynamicsLinear
from hcnn.training import DropoutMLP

In [2]:
import jax

jax.config.update("jax_cuda_visible_devices", "0")
# jax.config.update("jax_platform_name", "cpu")

jax.devices()

[CudaDevice(id=0)]

In [3]:
state_matrix = jnp.array([[1.0, -1.0], [1.0, 0.0]])
input_matrix = jnp.array([[1.0], [0.0]])
sys_dynamics = SystemDynamicsLinear.factory(state_matrix, input_matrix)

In [4]:
sys_constraints = SystemConstraintsBox.factory(
    state_l=jnp.array([-3.0, -3.0]),
    state_u=jnp.array([3.0, 3.0]),
    action_l=jnp.array([-jnp.inf]),
    action_u=jnp.array([jnp.inf]),
)

In [5]:
state_cost = jnp.eye(2)
action_cost = jnp.eye(1)
sys_cost = SystemCostQuadratic.factory(state_cost, action_cost)

In [6]:
policy = PolicyMLP.factory(
    state_dim=sys_dynamics.state_dim,
    action_dim=sys_dynamics.action_dim,
    hidden_layer_sizes=[4, 4],
    dropout_rate=0.15,
    rngs=nnx.Rngs(42),
)

In [7]:
problem, phi = ProjectionNoSubstitutionNoTerminal.factory(
    policy=policy,
    sys_dynamics=sys_dynamics,
    sys_constraints=sys_constraints,
    horizon=20,
    relaxation_penalty=7.0,
)

In [8]:
solver = OptimizationSolverShallow(max_iter=50)

In [9]:
phi = phi.replace(initial_state=jnp.array([1.0, 2.0]))

In [10]:
%time x = problem.get_x0(phi).block_until_ready()

CPU times: user 1.75 s, sys: 208 ms, total: 1.96 s
Wall time: 4.6 s


In [11]:
%timeit problem.get_x0(phi).block_until_ready()

2.76 ms ± 356 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [12]:
%time c = problem.constraint_fn(x, phi).block_until_ready()

CPU times: user 1.89 s, sys: 59.4 ms, total: 1.95 s
Wall time: 4.41 s


In [13]:
%timeit problem.constraint_fn(x, phi).block_until_ready()

4.15 ms ± 695 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [14]:
%time out = solver.solve(problem, phi).block_until_ready()

CPU times: user 13.3 s, sys: 244 ms, total: 13.5 s
Wall time: 28.2 s


In [15]:
%timeit solver.solve(problem, phi).block_until_ready()

426 ms ± 37.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
def fun(initial_state):
    return solver.solve(problem, phi.replace(initial_state=initial_state))


jit_fn = jax.vmap(fun)

input = jax.random.normal(jax.random.key(4), (20, 2))
jit_fn(input).block_until_ready()

%timeit jit_fn(input).block_until_ready()

318 ms ± 78.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## ---  fix above first

In [ ]:
value_to_go = DropoutMLP(
    layer_sizes=[sys_dynamics.state_dim, 4, 1],
    dropout_rate=0.15,
    rngs=nnx.Rngs(10),
)

In [ ]:
relax_to_go = DropoutMLP(
    layer_sizes=[sys_dynamics.state_dim, 4, 1],
    dropout_rate=0.15,
    rngs=nnx.Rngs(10),
)

In [ ]:
environment = Environment(
    ## SYSTEM DEFINITION
    sys_constraints=sys_constraints,
    sys_cost=sys_cost,
    sys_dynamics=sys_dynamics,
    ## PROJECTION
    projection=projection,
    solver=solver,
)

In [ ]:
graphdef, state = environment.init(policy, value_to_go, relax_to_go, nnx.Rngs(42))

In [ ]:
for i in range(100):
    print(f"Iteration {i}")
    state = environment.train_step(graphdef, state)

In [ ]:
(
    environment,
    policy,
    policy_optimizer,
    value_to_go,
    value_to_go_optimizer,
    relax_to_go,
    relax_to_go_optimizer,
    metrics,
    rngs,
) = nnx.merge(graphdef, state)

In [ ]:
view_range = 6.0
samples_per_axis = 1_000
axis_samples = jnp.linspace(-view_range, view_range, samples_per_axis)
X, Y = jnp.meshgrid(axis_samples, axis_samples, indexing="xy")
states = jnp.stack((X.flatten(), Y.flatten()), axis=-1)

value_to_go.eval()
costs = nnx.vmap(lambda state: value_to_go(state, rngs=nnx.Rngs(0)))(
    states,
)
costs = costs.squeeze().reshape(samples_per_axis, samples_per_axis)

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(
    states[:, 0].reshape(samples_per_axis, samples_per_axis),
    states[:, 1].reshape(samples_per_axis, samples_per_axis),
    costs,
    levels=20,
    cmap="viridis",
)
rect = plt.Rectangle((-3, -3), 6, 6, linewidth=2, edgecolor="red", facecolor="none")
ax.add_patch(rect)
fig.colorbar(c, ax=ax, label="value-to-go")
ax.set_xlabel("State dimension 1")
ax.set_ylabel("State dimension 2")
ax.set_title("value-to-go over state")
plt.show()

In [ ]:
relax_to_go.eval()
costs = nnx.vmap(lambda state: relax_to_go(state, rngs=nnx.Rngs(0)))(
    states,
)
costs = costs.squeeze().reshape(samples_per_axis, samples_per_axis)

fig, ax = plt.subplots(figsize=(8, 6))
c = ax.contourf(
    states[:, 0].reshape(samples_per_axis, samples_per_axis),
    states[:, 1].reshape(samples_per_axis, samples_per_axis),
    costs,
    levels=20,
    cmap="viridis",
)
rect = plt.Rectangle((-3, -3), 6, 6, linewidth=2, edgecolor="red", facecolor="none")
ax.add_patch(rect)
fig.colorbar(c, ax=ax, label="relax-to-go")
ax.set_xlabel("State dimension 1")
ax.set_ylabel("State dimension 2")
ax.set_title("relax-to-go over state")
plt.show()